# Trading Algorithmique : Backtesting avec la librairie Backtrader

**Auteur :** ISLEYEN Volkan
**Statut :** Étudiant L3 Économie & Gestion | Candidat Master Finance de Marché
**Sujet :** Simulation historique d'une stratégie de croisement de moyennes mobiles
**Stack Technique :** Python, Backtrader, Pandas, Yfinance

---

## Contexte du Projet

La création d'indicateurs techniques n'est que la première étape de l'analyse quantitative. Pour valider la pertinence d'une stratégie sur les marchés financiers, il est impératif de la soumettre à l'épreuve des données historiques. 

Ce module est dédié au **backtesting**. L'objectif est de simuler les conditions réelles de marché (gestion du capital, exécution des ordres) pour évaluer la robustesse d'une stratégie algorithmique avant tout déploiement de capitaux. Pour ce faire, nous passons de la simple manipulation de données sur `pandas` à l'implémentation d'un véritable moteur d'exécution orienté objet : le framework **Backtrader**.

## La Stratégie Modélisée : SMA Crossover

La stratégie implémentée repose sur un grand classique du suivi de tendance : le **croisement de moyennes mobiles simples (SMA 10 jours et SMA 30 jours)**. 

Le modèle algorithmique génère des signaux selon les règles suivantes :
* **Signal d'Achat (Long) :** Lorsque la moyenne mobile courte (10 jours) croise à la hausse la moyenne mobile longue (30 jours). Cela traduit une accélération haussière du momentum.
* **Signal de Vente (Clôture de position) :** Lorsque la moyenne mobile courte croise à la baisse la moyenne mobile longue, marquant l'épuisement de la tendance.

## Évaluation de la Performance et du Risque (KPIs)

Une stratégie algorithmique ne s'évalue pas uniquement sur son profit brut, mais sur le risque assumé pour générer ce profit. À l'issue de la simulation via le module `Cerebro` de Backtrader, le script calcule les indicateurs de performance clés (KPIs) suivants :

* **Rendement Total :** Évolution du capital (à partir d'une dotation initiale de 10 000 $).
* **Maximum Drawdown (Max DD) :** La plus forte baisse historique du portefeuille par rapport à son pic. C'est l'indicateur fondamental de la gestion des risques.
* **Taux de Réussite (Win Rate) :** La proportion de transactions clôturées avec une plus-value.
* **Ratio Gain/Perte (Risk/Reward) :** Le gain moyen par trade gagnant pondéré par la perte moyenne par trade perdant.

## Architecture du Code

Le projet exploite l'architecture spécifique de `backtrader`. La logique de trading est encapsulée dans une **Classe** Python, ce qui démontre une maîtrise de la programmation orientée objet (OOP), une compétence technique particulièrement recherchée dans les profils de type "Quant".

In [3]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Télécharger les données Apple
df = yf.download('AAPL', start='2020-01-01', end='2024-12-31')


# Ça fonctionne même si yfinance renvoie un DataFrame multi-niveaux
df.columns = df.columns.droplevel('Ticker')
df.dropna(inplace=True)
df.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,72.400513,72.460776,71.156674,71.409778,135480400
2020-01-03,71.696632,72.455950,71.472454,71.629138,146322800
2020-01-06,72.267944,72.306514,70.568518,70.819216,118387200
2020-01-07,71.928062,72.533103,71.708703,72.277586,108872000
2020-01-08,73.085106,73.386423,71.631552,71.631552,132079200


In [4]:
# 1. Calcul des moyennes mobiles
df['SMA10'] = df['Close'].rolling(window=10).mean()
df['SMA30'] = df['Close'].rolling(window=30).mean()

In [6]:
!pip install backtrader --quiet 

In [9]:
import backtrader as bt
# On ne garde que les colonnes nécessaires pour le trading
data = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
data.dropna(inplace=True)  # On supprime les lignes incomplètes

# Backtrader préfère les noms de colonnes en minuscules
data.columns = [col.lower() for col in data.columns]

# On remet l'index 'Date' sous forme de colonne pour Backtrader
data.reset_index(inplace=True)
data.reset_index(inplace=True)
data.head()

,index,Date,open,high,low,close,volume
0,0,2020-01-02,71.409778,72.460776,71.156674,72.400513,135480400
1,1,2020-01-03,71.629138,72.455950,71.472454,71.696632,146322800
2,2,2020-01-06,70.819216,72.306514,70.568518,72.267944,118387200
3,3,2020-01-07,72.277586,72.533103,71.708703,71.928062,108872000
4,4,2020-01-08,71.631552,73.386423,71.631552,73.085106,132079200


In [17]:
# Adapter les données pour Backtrader
class PandasData(bt.feeds.PandasData):
    params = (
        ('datetime', 'Date'),
        ('open', 'open'),
        ('high', 'high'),
        ('low', 'low'),
        ('close', 'close'),
        ('volume', 'volume'),
        ('openinterest', -1),
    )

# Stratégie SMA(10) / SMA(30)
class SMACross(bt.Strategy):
    def __init__(self):
        self.sma1 = bt.ind.SMA(period=10)  # SMA courte
        self.sma2 = bt.ind.SMA(period=30)  # SMA longue
        self.cross = bt.ind.CrossOver(self.sma1, self.sma2)
        self.portfolio_values = []
        self.trades = []

    def next(self):
        self.portfolio_values.append(self.broker.getvalue())

        if not self.position:
            if self.cross > 0:
                self.buy_price = self.data.close[0]
                self.buy()
        elif self.cross < 0:
            if self.position:
                sell_price = self.data.close[0]
                gain = sell_price - self.buy_price
                self.trades.append(gain)
                self.sell()

    def stop(self):
        capital_initial = cerebro.startingcash
        capital_final = self.broker.getvalue()
        rendement = (capital_final - capital_initial) / capital_initial * 100

        peak = self.portfolio_values[0]
        max_dd = 0
        for val in self.portfolio_values:
            if val > peak:
                peak = val
            dd = (peak - val) / peak
            if dd > max_dd:
                max_dd = dd

        total_trades = len(self.trades)
        winning_trades = len([t for t in self.trades if t > 0])
        losing_trades = total_trades - winning_trades
        win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0
        avg_gain = sum(self.trades) / total_trades if total_trades > 0 else 0
        gains = [t for t in self.trades if t > 0]
        losses = [t for t in self.trades if t < 0]
        avg_win = sum(gains) / len(gains) if gains else 0
        avg_loss = abs(sum(losses) / len(losses)) if losses else 0
        gain_loss_ratio = avg_win / avg_loss if avg_loss > 0 else 0

        print("----- Résultats de la stratégie SMA(10)/SMA(30) -----")
        print(f"Capital initial     : {capital_initial:.2f} $")
        print(f"Capital final       : {capital_final:.2f} $")
        print(f"Rendement total     : {rendement:.2f} %")
        print(f"Drawdown maximal    : {max_dd * 100:.2f} %")
        print(f"Nombre de trades    : {total_trades}")
        print(f"Trades gagnants     : {winning_trades}")
        print(f"Trades perdants     : {losing_trades}")
        print(f"Taux de réussite    : {win_rate:.2f} %")
        print(f"Gain moyen / trade  : {avg_gain:.2f} $")
        print(f"Ratio gain/perte    : {gain_loss_ratio:.2f}")

# Lancer le backtest
cerebro = bt.Cerebro()
cerebro.addstrategy(SMACross)
cerebro.adddata(PandasData(dataname=data))
cerebro.broker.setcash(10000.0)
cerebro.startingcash = 10000.0

cerebro.run()
cerebro.plot(iplot=False, style='candlestick', volume=True)

----- Résultats de la stratégie SMA(10)/SMA(30) -----
Capital initial     : 10000.00 $
Capital final       : 10102.51 $
Rendement total     : 1.03 %
Drawdown maximal    : 0.45 %
Nombre de trades    : 20
Trades gagnants     : 9
Trades perdants     : 11
Taux de réussite    : 45.00 %
Gain moyen / trade  : 3.93 $
Ratio gain/perte    : 2.31


[[<Figure size 485x356 with 5 Axes>]]